# Redes de personagens em Cyberpunk: Edgerunners (temporada 1)
45RPE: Resolução de Problemas com Estruturas de Dados, trabalho de recuperação.

**Arquivos necessários na mesma pasta do notebook:** `base_interacoes.txt`.

## 1. Identificação da equipe
**PENDENTE:** integrantes da equipe (até 3 estudantes).

## 2. Introdução e objetivos
**Pergunta central:** como a estrutura de poder (Arasaka e entorno corporativo) já existia e absorveu David, criando o terreno para as traições.

- Rede principal centrada em David, com os personagens secundários conectados entre si.
- Análise com evolução episódio por episódio.

**PENDENTE:** redação final da pergunta central e a pergunta e o objetivo de cada análise.

## 3. Descrição da obra e delimitação do corpus
**Obra:** Cyberpunk: Edgerunners, temporada 1 (Netflix, 10 episódios), que cumpre o mínimo de extensão de série.

**Contexto:** futuro distópico com monopólio de empresas de tecnologia e hardware, envolvendo pessoas de camadas sociais diferentes. David é matriculado pela mãe numa escola de alto padrão de onde pessoas são recrutadas pela Arasaka.

**PENDENTE:**
- Investigação bibliográfica de ineditismo, documentada.
- Justificativa da escolha da obra.
- Aprovação do docente.

## 4. Definição do esquema de dados
**Registro:** `personagem1 personagem2 sinal ambiente episodio minuto`, campos separados por um espaço em branco. A linha logo abaixo, iniciada por `#`, traz a descrição breve do evento, que também serve de justificativa do sinal.

**Interação:** conversa, ação dirigida ao outro (incluindo conflitos) ou ação conjunta em que os dois agem juntos por um mesmo objetivo. Chamada em que os dois se respondem conta como interação direta; interação indireta não entra. Se uma pessoa dá uma ordem a vários e só um responde, é um registro.

**Início e fim:** a cada mudança de cena ou de ambiente; quando um dos personagens sai da cena, a interação termina.

**Conversa prolongada e eventos distintos:** um registro por cena, mesmo com várias falas. Cenas diferentes do mesmo par geram registros separados. Se a mesma cena muda de sinal, são dois registros do mesmo par e ambiente, com minutos diferentes.

**Sinal**, pelo efeito da ação sobre o outro personagem, de forma óbvia:
- `+`: a ação ajuda o outro.
- `-`: a ação prejudica o outro (agressão é negativa mesmo que a vítima merecesse).
- `0`: interação direta que não ajuda nem prejudica de forma clara.
- Sinal incerto: o caso fica de fora das análises de sinal.

**Identificadores:** `NomeSobrenome`, sem acento, espaço ou símbolo, conforme a seção 4.1 do enunciado. Personagem sem nome recebe um ID descritivo (ex.: `DiretorAcademia`). O dicionário (ID, nome canônico, apelidos e primeira aparição) está no cabeçalho de `base_interacoes.txt`.

## 5. Descrição da coleta e validação das interações
**Coleta:** a série foi assistida episódio por episódio, com áudios curtos transcritos que funcionaram como diário.

A célula abaixo lê a base (dicionário e registros) e a próxima valida cada registro.

In [1]:
import re
from pathlib import Path

import pandas as pd
import networkx as nx  # usado nas seções de modelagem e análises

BASE = Path("base_interacoes.txt")

personagens = {}   # id -> dados do dicionário
registros = []     # um item por registro de interação

for n, linha in enumerate(BASE.read_text(encoding="utf-8").splitlines(), start=1):
    linha = linha.strip()
    if not linha:
        continue
    if linha.startswith("# PERSONAGEM "):
        id_, nome, apelidos, primeira = [c.strip() for c in linha[len("# PERSONAGEM "):].split("|")]
        personagens[id_] = {
            "nome_canonico": nome,
            "apelidos": [a.strip() for a in apelidos.split(";") if a.strip()],
            "primeira_aparicao": primeira,
        }
    elif linha.startswith("#"):
        # a linha de comentário logo abaixo de um registro é a descrição dele
        if registros and registros[-1]["linha"] == n - 1 and not registros[-1]["descricao"]:
            registros[-1]["descricao"] = linha[1:].strip()
    else:
        registros.append({"linha": n, "campos": linha.split(" "), "descricao": ""})

print(f"{len(personagens)} personagens no dicionário e {len(registros)} registros lidos")

19 personagens no dicionário e 15 registros lidos


In [2]:
padrao_id = re.compile(r"^[A-Za-z0-9]+$")
padrao_episodio = re.compile(r"^E\d{2}$")
padrao_minuto = re.compile(r"^\d{2}:\d{2}$")

erros = []
linhas_validas = []
for r in registros:
    c = r["campos"]
    if len(c) != 6:
        erros.append((r["linha"], f"esperados 6 campos, encontrados {len(c)}"))
        continue
    p1, p2, sinal, ambiente, episodio, minuto = c
    for p in (p1, p2):
        if not padrao_id.match(p):
            erros.append((r["linha"], f"ID inválido: {p}"))
        if p not in personagens:
            erros.append((r["linha"], f"ID fora do dicionário: {p}"))
    if p1 == p2:
        erros.append((r["linha"], "personagem1 e personagem2 são iguais"))
    if sinal not in ("+", "-", "0"):
        erros.append((r["linha"], f"sinal inválido: {sinal}"))
    if not padrao_id.match(ambiente):
        erros.append((r["linha"], f"ambiente inválido: {ambiente}"))
    if not padrao_episodio.match(episodio):
        erros.append((r["linha"], f"episódio inválido: {episodio}"))
    if not padrao_minuto.match(minuto):
        erros.append((r["linha"], f"minuto inválido: {minuto}"))
    if not r["descricao"]:
        erros.append((r["linha"], "registro sem descrição"))
    linhas_validas.append({
        "linha": r["linha"], "personagem1": p1, "personagem2": p2, "sinal": sinal,
        "ambiente": ambiente, "episodio": episodio, "minuto": minuto,
        "descricao": r["descricao"],
    })

assert not erros, erros
df = pd.DataFrame(linhas_validas)
print(f"Base válida: {len(df)} registros")

Base válida: 15 registros


## 6. Estatísticas descritivas da base

In [3]:
participam = set(df["personagem1"]) | set(df["personagem2"])
print(f"Registros: {len(df)}")
print(f"Personagens no dicionário: {len(personagens)}; com ao menos um registro: {len(participam)}")
print()
print("Registros por sinal:")
print(df["sinal"].value_counts().to_string())
print()
print("Registros por episódio e sinal:")
print(df.groupby(["episodio", "sinal"]).size().unstack(fill_value=0).to_string())
print()
print("Registros por ambiente:")
print(df["ambiente"].value_counts().to_string())

Registros: 15
Personagens no dicionário: 19; com ao menos um registro: 11

Registros por sinal:
sinal
-    11
+     3
0     1

Registros por episódio e sinal:
sinal     +   -  0
episodio          
E01       3  11  1

Registros por ambiente:
ambiente
ApartamentoDavid    6
PatioArasaka        2
AcademiaArasaka     2
Hospital            2
Rodovia             1
CentroDaCidade      1
OficinaRipperdoc    1


## 7. Modelagem das redes
Cada vértice é um personagem e cada aresta liga dois personagens que interagiram. O peso da aresta é o número de registros do par, ou seja, em quantas cenas eles interagiram, e as interações repetidas somam nesse peso. A rede é não direcionada, porque a pergunta trata de quem se conecta a quem e de como o sinal muda, e não de quem age sobre quem. A limitação é que a rede não mostra o sentido da ação nem a reciprocidade.

O sinal entra em redes separadas: uma só com as interações positivas e outra só com as negativas, além da rede global com todas as interações. Um par com sinais diferentes aparece nas duas, cada uma com o seu peso. Os casos de sinal incerto ficam fora das análises de sinal.

A rede principal é centrada no David, com os personagens secundários conectados entre si, e também montamos redes por episódio para acompanhar a evolução.

## 8. Análises
**PENDENTE:** pelo menos oito análises distintas, com pelo menos três visualizações e cinco métricas ou algoritmos. Cada análise traz título, pergunta e objetivo, técnica, configuração, resultados, interpretação e conclusões.

## 9. Síntese dos resultados
**PENDENTE.**

## 10. Referências
- Apoorv Agarwal, Augusto Corvalan, Jacob Jensen, and Owen Rambow. Social network analysis of Alice in Wonderland. In Proceedings of the NAACL-HLT 2012 Workshop on Computational Linguistics for Literature, pages 88-96. Association for Computational Linguistics, 2012. https://aclanthology.org/W12-2513/
- Apoorv Agarwal, Sriramkumar Balasubramanian, Jiehan Zheng, and Sarthak Dash. Parsing screenplays for extracting social networks from movies. In Proceedings of the 3rd Workshop on Computational Linguistics for Literature, pages 50-58. Association for Computational Linguistics, 2014. doi: 10.3115/v1/W14-0907.
- Vincent Labatut and Xavier Bost. Extraction and analysis of fictional character networks: A survey. ACM Computing Surveys, 52(5):1-40, 2019. doi: 10.1145/3344548.